In [42]:
import os
import torch
import pandas as pd
import torch.nn as nn
import torch.optim as optim
#from tqdm.notebook import tqdm

from tqdm import tqdm
from PIL import Image
from torchvision import models
from torchvision import transforms
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torch.utils.data import WeightedRandomSampler

from sklearn.metrics import f1_score, accuracy_score

from utils.dataset import *
from utils.training import *

## Dataset + Dataloader

In [2]:
root = '/gpfs01/berens/data/data/medical_imaging/Dermoscopy/ISIC_2018/Task3/'
img_train_dir = os.path.join(root, 'Training_Input')
img_val_dir = os.path.join(root, 'Validation_Input')
img_test_dir = os.path.join(root, 'Test_Input')

In [33]:
test = False

df_train = pd.read_csv(os.path.join(root, 'Training_GroundTruth_v2.csv'))
if test==True:
    df_train = df_train.sample(500)
    
df_val = pd.read_csv(os.path.join(root, 'Validation_GroundTruth_v2.csv'))
df_test = pd.read_csv(os.path.join(root, 'Test_GroundTruth_v2.csv'))

print(len(df_train), len(df_val), len(df_test))

10015 193 1512


In [34]:
train_transform, val_transform = get_transform()

In [35]:
train_dset = ISICDataset(df_train, img_train_dir, train_transform)
val_dset = ISICDataset(df_val, img_val_dir, val_transform)
test_dset = ISICDataset(df_test, img_test_dir, val_transform)

print(len(train_dset), len(val_dset), len(test_dset))

10015 193 1512


In [36]:
sampler = get_sampler(df_train)

train_loader = DataLoader(
    train_dset,
    batch_size=32,
    sampler=sampler,   # replaces shuffle=True
    num_workers=4
)

val_loader = DataLoader(
    val_dset,
    batch_size=32,
    shuffle=False,
    num_workers=4
)

test_loader = DataLoader(
    test_dset,
    batch_size=32,
    shuffle=False,
    num_workers=4
)

print(len(train_loader), len(val_loader), len(test_loader))

313 7 48


## Training loop

In [37]:
xai = False
model = Explainable_model(network='resnet50', num_classes=7) if xai else build_model(num_classes=7, backbone='resnet50')

In [38]:
criterion = nn.CrossEntropyLoss()  # no weight= needed since we're using the sampler
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

In [39]:
# training loop
best_f1 = 0.0
num_epochs = 20
t_acc, t_loss, t_f1, v_acc, v_loss, v_f1 = [], [], [], [], [], []
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu') 


pbar = tqdm(range(num_epochs), desc="Epochs")

for epoch in pbar:
    train_loss, train_preds, train_labels = train_one_epoch(model, train_loader, criterion, optimizer, device, xai)
    train_f1 = f1_score(train_labels, train_preds, average='macro')
    train_acc = accuracy_score(train_labels, train_preds)
    t_loss.append(train_loss), t_acc.append(train_acc), t_f1.append(train_f1)
    
    val_loss, val_preds, val_labels = evaluate(model, val_loader, criterion, device, xai)
    val_f1 = f1_score(val_labels, val_preds, average='macro')
    val_acc = accuracy_score(val_labels, val_preds)
    v_loss.append(val_loss), v_acc.append(val_acc), v_f1.append(val_f1)
    
    scheduler.step()

    pbar.set_postfix(train_loss=f"{train_loss:.3f}",  train_acc=f"{train_acc:.3f}", train_f1=f"{train_f1:.3f}", val_loss=f"{val_loss:.3f}", vall_acc=f"{val_acc:.3f}", val_f1=f"{val_f1:.3f}")
    #print(f"Epoch {epoch+1}/{num_epochs} | train_loss={train_loss:.3f}, train_acc={train_acc:.3f}, train_f1={train_f1:.3f}, val_loss={val_loss:.3f}, vall_acc={val_acc:.3f}, val_f1={val_f1:.3f}")

    data = pd.DataFrame({'train_loss': t_loss, 'train_acc': t_acc, 'train_f1': t_f1, 'val_loss': v_loss, 'val_acc': v_acc, 'val_f1': v_f1})
    
    if val_f1 > best_f1:
        best_f1 = val_f1
        if xai:
            torch.save(model.state_dict(), 'save/best_expl_model.pth')
            data.to_csv('save/training_xai.csv', index=False)
        else:
            torch.save(model.state_dict(), 'save/best_model.pth')
            data.to_csv('save/training.csv', index=False)
        print(f"  -> Saved new best model (F1: {best_f1:.4f})")

Epochs:   5%|▌         | 1/20 [02:44<52:08, 164.67s/it, train_acc=0.695, train_f1=0.693, train_loss=0.846, val_f1=0.711, val_loss=0.629, vall_acc=0.798]

  -> Saved new best model (F1: 0.7110)


Epochs:  10%|█         | 2/20 [05:29<49:23, 164.63s/it, train_acc=0.844, train_f1=0.844, train_loss=0.441, val_f1=0.757, val_loss=0.573, vall_acc=0.813]

  -> Saved new best model (F1: 0.7568)


Epochs:  15%|█▌        | 3/20 [08:14<46:41, 164.77s/it, train_acc=0.886, train_f1=0.885, train_loss=0.316, val_f1=0.770, val_loss=0.481, vall_acc=0.850]

...

  -> Saved new best model (F1: 0.8385)


Epochs:  80%|████████  | 16/20 [44:10<11:02, 165.67s/it, train_acc=0.987, train_f1=0.987, train_loss=0.040, val_f1=0.840, val_loss=0.456, vall_acc=0.902]

  -> Saved new best model (F1: 0.8402)


Epochs: 100%|██████████| 20/20 [55:12<00:00, 165.61s/it, train_acc=0.992, train_f1=0.991, train_loss=0.030, val_f1=0.846, val_loss=0.489, vall_acc=0.907]

  -> Saved new best model (F1: 0.8456)


In [43]:
xai = True
model = Explainable_model(network='resnet50', num_classes=7) 

criterion = nn.CrossEntropyLoss()  # no weight= needed since we're using the sampler
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

In [44]:
# training loop
best_f1 = 0.0
num_epochs = 40
t_acc, t_loss, t_f1, v_acc, v_loss, v_f1 = [], [], [], [], [], []
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu') 

pbar = tqdm(range(num_epochs), desc="Epochs")

for epoch in pbar:
    train_loss, train_preds, train_labels = train_one_epoch(model, train_loader, criterion, optimizer, device, xai)
    train_f1 = f1_score(train_labels, train_preds, average='macro')
    train_acc = accuracy_score(train_labels, train_preds)
    t_loss.append(train_loss), t_acc.append(train_acc), t_f1.append(train_f1)
    
    val_loss, val_preds, val_labels = evaluate(model, val_loader, criterion, device, xai)
    val_f1 = f1_score(val_labels, val_preds, average='macro')
    val_acc = accuracy_score(val_labels, val_preds)
    v_loss.append(val_loss), v_acc.append(val_acc), v_f1.append(val_f1)
    
    scheduler.step()

    pbar.set_postfix(train_loss=f"{train_loss:.3f}",  train_acc=f"{train_acc:.3f}", train_f1=f"{train_f1:.3f}", val_loss=f"{val_loss:.3f}", vall_acc=f"{val_acc:.3f}", val_f1=f"{val_f1:.3f}")
    #print(f"Epoch {epoch+1}/{num_epochs} | train_loss={train_loss:.3f}, train_acc={train_acc:.3f}, train_f1={train_f1:.3f}, val_loss={val_loss:.3f}, vall_acc={val_acc:.3f}, val_f1={val_f1:.3f}")

    data = pd.DataFrame({'train_loss': t_loss, 'train_acc': t_acc, 'train_f1': t_f1, 'val_loss': v_loss, 'val_acc': v_acc, 'val_f1': v_f1})
    
    if val_f1 > best_f1:
        best_f1 = val_f1
        if xai:
            torch.save(model.state_dict(), 'save/best_expl_model.pth')
            data.to_csv('save/training_xai.csv', index=False)
        else:
            torch.save(model.state_dict(), 'save/best_model.pth')
            data.to_csv('save/training.csv', index=False)
            
        print(f"  -> Saved new best model (F1: {best_f1:.4f})")

Epochs:   2%|▎         | 1/40 [02:45<1:47:23, 165.21s/it, train_acc=0.701, train_f1=0.697, train_loss=0.838, val_f1=0.640, val_loss=0.715, vall_acc=0.777]

  -> Saved new best model (F1: 0.6405)


Epochs:   5%|▌         | 2/40 [05:32<1:45:19, 166.29s/it, train_acc=0.845, train_f1=0.846, train_loss=0.432, val_f1=0.742, val_loss=0.502, vall_acc=0.855]

  -> Saved new best model (F1: 0.7425)


Epochs:   8%|▊         | 3/40 [08:17<1:42:17, 165.88s/it, train_acc=0.890, train_f1=0.889, train_loss=0.309, val_f1=0.771, val_loss=0.457, vall_acc=0.860]

...

Epochs:  52%|█████▎    | 21/40 [57:56<52:32, 165.90s/it, train_acc=0.992, train_f1=0.992, train_loss=0.029, val_f1=0.829, val_loss=0.503, vall_acc=0.891]

  -> Saved new best model (F1: 0.8287)


Epochs:  85%|████████▌ | 34/40 [1:33:40<16:32, 165.50s/it, train_acc=0.977, train_f1=0.977, train_loss=0.074, val_f1=0.841, val_loss=0.558, vall_acc=0.891]

  -> Saved new best model (F1: 0.8406)


Epochs: 100%|██████████| 40/40 [1:50:14<00:00, 165.36s/it, train_acc=0.975, train_f1=0.976, train_loss=0.074, val_f1=0.834, val_loss=0.735, vall_acc=0.824]


## ToDo
- Balance accuracy
- Improve classification performance? Which HP?
- Add sparsity constraint on the interpretable model
- Implement others backbones?